In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import rc
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
import imageio.v2 as imageio
from PIL import Image

#########################################
# Function: calc_parc_o3
#########################################
def calc_parc_o3(var, p_top=30, p_bottom=100):
    """
    Calculate the partial ozone column for a specified pressure range.
    
    Parameters:
        var: xarray.DataArray
            Ozone data (units: mol/mol)
        p_top: float
            Top pressure level (hPa)
        p_bottom: float
            Bottom pressure level (hPa)
    
    Returns:
        xarray.DataArray:
            Ozone column in Dobson Units (DU)
    """
    m_air = 28.964 / (6.022E23)
    g = 980.6

    # Determine pressure coordinate name (plev, lev, or level)
    if 'plev' in var.dims:
        plev = var.plev
        level = 'plev'
    elif 'lev' in var.dims:
        plev = var.lev
        level = 'lev'
    elif 'level' in var.dims:
        plev = var.level
        level = 'level'
    else:
        raise ValueError("Pressure coordinate not found: plev/lev/level")

    time = var.time
    delta_p = np.zeros((len(time), len(plev)))

    # Determine conversion factors for pressure coordinate
    if plev[0] < plev[-1] and plev[-1] <= 1000:
        factor = 100  # convert hPa to Pa
        factor_2 = 1
    elif plev[0] > plev[-1] and plev[0] <= 1000:
        factor = 100
        factor_2 = 1
    elif plev[0] < plev[-1] and plev[-1] > 1000:
        factor = 1
        factor_2 = 100
    elif plev[0] > plev[-1] and plev[0] > 1000:
        factor = 1
        factor_2 = 100
    else:
        factor = 1
        factor_2 = 1

    # If pressure levels are in ascending order
    if plev[0] < plev[-1]:
        for i in range(1, len(plev)):
            delta_p[:, i] = plev[i] - plev[i-1]
        weights_p = xr.DataArray(delta_p * factor, dims=['time', level], coords=[time, plev])
        O3 = var * weights_p * 10 / (g * m_air)
        # Select layers between p_top and p_bottom
        O3 = O3.sel(**{level: slice(p_top * factor_2, p_bottom * factor_2)})
        O3 = O3.sum(dim=level)
        O3 = O3 / 2.687E16  # convert to DU
    else:
        for i in range(len(plev)-1):
            delta_p[:, i] = plev[i] - plev[i+1]
        weights_p = xr.DataArray(delta_p * factor, dims=['time', level], coords=[time, plev])
        O3 = var * weights_p * 10 / (g * m_air)
        O3 = O3.sel(**{level: slice(p_bottom * factor_2, p_top * factor_2)})
        O3 = O3.sum(dim=level)
        O3 = O3 / 2.687E16  # convert to DU

    return O3.where(O3 != 0)

#########################################
# Function: compute_area_average
#########################################
def compute_area_average(field):
    """
    Compute the area-average of a field over lat and lon.
    
    If the field has a 'lon' dimension with more than one value, first average over lon,
    then compute the cosine–weighted average over lat.
    """
    if 'lon' in field.dims and field.lon.size > 1:
        field_avg = field.mean(dim='lon')
    else:
        field_avg = field
    if 'lat' in field_avg.dims:
        weights = np.cos(np.deg2rad(field_avg.lat))
        field_avg = field_avg.weighted(weights).mean(dim='lat')
    return field_avg

#########################################
# Function: load_experiment_field
#########################################
def load_experiment_field(file_pattern, p_top, p_bottom, lat_min=60, lat_max=90):
    """
    Load and process ozone data from the given file pattern for mapping.
    This function computes the partial ozone column over the specified pressure range,
    subsets the latitude range, and averages over ensemble members if applicable.
    
    Parameters:
        file_pattern: str
            Path pattern for NetCDF files.
        p_top, p_bottom: float
            Pressure range in hPa.
        lat_min, lat_max: float
            Latitude bounds (in degrees).
    
    Returns:
        xarray.DataArray:
            Processed ozone field with dimensions (time, lat, lon)
    """
    ds = xr.open_mfdataset(file_pattern, concat_dim='member', combine='nested')
    o3 = ds['O3']
    o3_col = calc_parc_o3(o3, p_top, p_bottom)
    o3_field = o3_col.sel(lat=slice(lat_min, lat_max))
    if 'member' in o3_field.dims:
        o3_field = o3_field.mean(dim='member')
    return o3_field

#########################################
# Functions: Load Experiment Data for Nocouple and Small Perturbation
#########################################
def load_year_nocouple_field(year, p_top, p_bottom, lat_min=60, lat_max=90):
    """
    Load February and March nocouple data for a given year for mapping.
    """
    file_Feb = f'/home/weiji/restart_exam/nocouple_data/{year}/Feb_NOCOUPL/{year}-02/*.nc'
    file_Mar = f'/home/weiji/restart_exam/nocouple_data/{year}/Mar_NOCOUPL/{year}-03/*.nc'
    data_Feb = load_experiment_field(file_Feb, p_top, p_bottom, lat_min, lat_max)
    data_Mar = load_experiment_field(file_Mar, p_top, p_bottom, lat_min, lat_max)
    return data_Feb, data_Mar

def load_year_small_pert_field(year, p_top, p_bottom, lat_min=60, lat_max=90):
    """
    Load February and March small perturbation data for a given year for mapping.
    """
    base_path = f'/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{year}'
    file_Feb = os.path.join(base_path, 'Feb', f'BWCN.e122.f19_g16.002.{year}-02.*.nc')
    file_Mar = os.path.join(base_path, 'Mar', f'BWCN.e122.f19_g16.002.{year}-03.*.nc')
    data_Feb = load_experiment_field(file_Feb, p_top, p_bottom, lat_min, lat_max)
    data_Mar = load_experiment_field(file_Mar, p_top, p_bottom, lat_min, lat_max)
    return data_Feb, data_Mar

#########################################
# Function: plot_composite_daily_maps
#########################################
def plot_composite_daily_maps(exp1_name, data1, exp2_name, data2, ref_field, output_dir, pressure_range, frame_duration=1000):
    """
    Plot daily polar maps for two experiments (difference relative to reference) side by side and save each frame as a PNG.
    The left panel shows experiment 1 difference (exp - ref) and the right panel shows experiment 2 difference (exp - ref).
    The color scale is fixed based on the pressure range.
    
    Parameters:
        exp1_name: str
            Name of experiment 1 (e.g., "nocoupl").
        data1: xarray.DataArray
            Ozone field for experiment 1 (dimensions: time, lat, lon).
        exp2_name: str
            Name of experiment 2 (e.g., "small").
        data2: xarray.DataArray
            Ozone field for experiment 2 (dimensions: time, lat, lon).
        ref_field: xarray.DataArray
            Reference ozone field (dimensions: time, lat, lon) for the corresponding year and pressure range.
        output_dir: str
            Directory to save PNG images.
        pressure_range: str
            Label for the pressure range (e.g., "4-20hPa").
        frame_duration: int
            Duration for each frame in the final GIF (milliseconds).
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Set up fonts for plotting
    rc('font', **{'family': 'sans-serif', 'sans-serif': ['Helvetica']})
    rc('text', usetex=False)
    
    # Fixed color limits based on pressure_range
    if pressure_range == "20-100hPa":
        vmin, vmax = -70, 70
    elif pressure_range == "4-20hPa":
        vmin, vmax = -15, 15
    elif pressure_range == "1-4hPa":
        vmin, vmax = -3, 3
    else:
        vmin, vmax = None, None  # fallback
    
    n_frames = min(data1.time.size, data2.time.size, ref_field.time.size)
    
    for i in range(n_frames):
        # Extract the i-th time slice for both experiments and the reference field
        field1 = data1.isel(time=i)
        field2 = data2.isel(time=i)
        ref_slice = ref_field.isel(time=i)
        
        # Compute difference with the reference field for this time step
        diff1 = field1 - ref_slice
        diff2 = field2 - ref_slice
        
        # Check if 'lon' coordinate exists and is non-empty
        if 'lon' in diff1.coords and diff1.lon.size > 0:
            diff1_vals, lons = add_cyclic_point(diff1.values, coord=diff1.lon.values)
            diff2_vals, _ = add_cyclic_point(diff2.values, coord=diff2.lon.values)
        else:
            print(f"Frame {i+1}: 'lon' coordinate is missing or empty, skipping this frame.")
            continue
        
        # Compute area averages for difference fields
        avg1 = float(compute_area_average(diff1).compute())
        avg2 = float(compute_area_average(diff2).compute())
        
        # Get time label from field1 (convert to numpy.datetime64)
        time_val = field1.time.values
        if isinstance(time_val, np.ndarray) and time_val.size == 1:
            time_val = time_val.item()
        time_val = np.datetime64(time_val)
        time_label = np.datetime_as_string(time_val, unit='D')
        
        # Create figure with two subplots (side by side) using fixed color scale
        fig, axes = plt.subplots(1, 2, subplot_kw={'projection': ccrs.NorthPolarStereo()}, figsize=(16,8))
        for ax in axes:
            ax.set_extent([-180, 180, 60, 90], ccrs.PlateCarree())
            ax.coastlines()
            ax.add_feature(cfeature.LAND, facecolor='lightgray')
            gl = ax.gridlines(draw_labels=True, linestyle='--', color='gray')
            gl.xlabels_top = False
            gl.ylabels_right = False
        
        # Left panel: experiment 1 difference
        cf1 = axes[0].contourf(lons, diff1.lat, diff1_vals, levels=np.linspace(vmin, vmax, 21),
                                 cmap='RdBu_r', extend='both', transform=ccrs.PlateCarree())
        axes[0].set_title(f"{exp1_name} (Diff Avg: {avg1:.2f} DU)\nDate: {time_label}", fontsize=14)
        # Right panel: experiment 2 difference
        cf2 = axes[1].contourf(lons, diff2.lat, diff2_vals, levels=np.linspace(vmin, vmax, 21),
                                 cmap='RdBu_r', extend='both', transform=ccrs.PlateCarree())
        axes[1].set_title(f"{exp2_name} (Diff Avg: {avg2:.2f} DU)\nDate: {time_label}", fontsize=14)
        
        # Add a common colorbar at the bottom
        cbar = fig.colorbar(cf2, ax=axes, orientation='horizontal', fraction=0.05, pad=0.08)
        cbar.set_label(f"Ozone Column Difference (DU) [{pressure_range}]", fontsize=12)
        
        fig.suptitle(f"Composite Ozone Difference (Exp - Ref)\nPressure Range: {pressure_range}", fontsize=16)
        
        out_fname = os.path.join(output_dir, f"frame_{i+1:03d}.png")
        plt.savefig(out_fname, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved frame {i+1} to {out_fname}")

#########################################
# Function: create_gif_from_directory
#########################################
def create_gif_from_directory(image_dir, output_gif, duration=1000):
    """
    Read PNG images (sorted by filename) from a directory and combine them into a GIF.
    
    Parameters:
        image_dir: str
            Directory containing PNG images.
        output_gif: str
            Output GIF file path.
        duration: int
            Duration between frames in milliseconds. now it's 1s already!
    """
    files = sorted([f for f in os.listdir(image_dir) if f.endswith('.png')])
    if not files:
        print(f"No PNG images found in {image_dir}!")
        return

    images = []
    for fname in files:
        fpath = os.path.join(image_dir, fname)
        try:
            images.append(imageio.imread(fpath))
        except Exception as e:
            print(f"Error reading {fpath}: {e}")
    imageio.mimsave(output_gif, images, duration=duration/1000)
    print(f"Saved GIF: {output_gif}")

#########################################
# Main Program
#########################################
if __name__ == "__main__":
    # Base output directory for composite analysis
    base_output_dir = "/home/weiji/restart_exam/20250221/2Danalysis"
    os.makedirs(base_output_dir, exist_ok=True)
    
    # Define experiment years, months, and pressure ranges
    years = ['0008', '0013', '0014', '0019']
    months = ['Feb', 'Mar']
    # Define pressure ranges as tuples: (p_top, p_bottom) and corresponding label strings.
    pressure_ranges = [((4, 20), "4-20hPa"), ((1, 4), "1-4hPa"), ((20, 100), "20-100hPa")]
    
    # Load the reference dataset once (contains all years)
    ref_path = "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002/BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc"
    ds_ref_all = xr.open_dataset(ref_path)
    ref_o3_all = ds_ref_all['O3']
    
    # Loop over each year and pressure range to extract reference data,
    # then process each month (Feb and Mar) for composite difference plots.
    for year in years:
        for (p_top, p_bottom), pr_label in pressure_ranges:
            print(f"\nProcessing Year: {year}, Pressure Range: {pr_label}")
            # Filter reference data for the current year and for months January–May
            ref_sel = ref_o3_all.sel(time=(ref_o3_all.time.dt.year == int(year)) &
                                     (ref_o3_all.time.dt.month.isin([2, 3, 4, 5]))).sortby('time')
            ref_col = calc_parc_o3(ref_sel, p_top, p_bottom)
            ref_col = ref_col.sel(lat=slice(60, 90))
            if 'member' in ref_col.dims:
                ref_mean = ref_col.mean(dim='member')
            else:
                ref_mean = ref_col
            # 注意：这里不再对时间做平均，保留时间维度
            
            for month in months:
                print(f"Processing Month: {month}")
                # Load experiment fields for the given year, month, and pressure range
                if month == 'Feb':
                    data_nocouple, _ = load_year_nocouple_field(year, p_top, p_bottom)
                    data_small, _ = load_year_small_pert_field(year, p_top, p_bottom)
                    # 筛选对应月份的参考场（假定时间步按月份对齐）
                    ref_field = ref_mean.sel(time=ref_mean.time.dt.month.isin([2, 3, 4, 5]))
                elif month == 'Mar':
                    _, data_nocouple = load_year_nocouple_field(year, p_top, p_bottom)
                    _, data_small = load_year_small_pert_field(year, p_top, p_bottom)
                    ref_field = ref_mean.sel(time=ref_mean.time.dt.month.isin([3, 4, 5]))
                else:
                    continue
                
                # Define output subdirectory for PNG frames for this combination
                combo_dir = os.path.join(base_output_dir, f"{year}_{month}_{pr_label}")
                os.makedirs(combo_dir, exist_ok=True)
                
                # Generate composite daily maps (difference: experiment - ref) for each time step
                plot_composite_daily_maps("nocoupl", data_nocouple, "small", data_small,
                                          ref_field, combo_dir, pr_label, frame_duration=500)
                
                # Create composite GIF from the generated PNG frames
                gif_fname = os.path.join(base_output_dir, f"composite_{year}_{month}_{pr_label}.gif")
                create_gif_from_directory(combo_dir, gif_fname, duration=500)
    
    print("\nAll composite GIFs have been generated!")
